# Week 12 — QuakeMigrate Travel-Time Lookup Tables
Confirmed directly from QuakeMigrate's own documentation: only a 1D
velocity model utility is currently implemented for LUT generation (not a
limitation we're working around - the tool's own supported usage pattern),
using the "1dnlloc" method (NonLinLoc's Grid2Time as backend, already
compiled and working since Week 6). Reuses the real Chow et al. (2022)
profile already extracted, and the same aeqd projection used consistently
throughout the project (GrowClust3D).

In [ ]:
import os
import pandas as pd
import obspy
from pyproj import Proj
from obspy.core import AttribDict

PROJECT_DIR = os.path.expanduser("~/tvz_project")
QM_DIR = os.path.join(PROJECT_DIR, "quakemigrate")
QM_LUT_DIR = os.path.join(QM_DIR, "lut")
os.makedirs(QM_LUT_DIR, exist_ok=True)

TARGET_LAT, TARGET_LON = -38.8, 176.0
NLLOC_BIN_DIR = os.path.join(PROJECT_DIR, "software", "NonLinLoc", "src", "bin")

In [ ]:
# Station file: Name, Longitude, Latitude, Elevation (case-sensitive, real
# 60-station inventory used throughout the project). Elevation in km,
# positive up (QuakeMigrate's convention - opposite sign to NonLinLoc's
# GTSRCE elevation convention used in Weeks 6/9, worth noting clearly).
INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")
WAVEFORM_DIR = os.path.join(PROJECT_DIR, "data", "waveforms")
real_stations = {d.split(".")[1] for d in os.listdir(WAVEFORM_DIR) if d.startswith("NZ.")}

inventory = obspy.read_inventory(INVENTORY_FILE)
rows = []
seen = set()
for net in inventory:
    for sta in net:
        if sta.code not in real_stations or sta.code in seen:
            continue
        rows.append({"Name": sta.code, "Longitude": sta.longitude,
                     "Latitude": sta.latitude, "Elevation": sta.elevation / 1000.0})
        seen.add(sta.code)

stations_df = pd.DataFrame(rows)
stations_path = os.path.join(QM_LUT_DIR, "stations.csv")
stations_df.to_csv(stations_path, index=False)
print(f"Saved: {stations_path} ({len(stations_df)} stations)")
print(stations_df.head())

In [ ]:
# Velocity model: Depth, Vp, Vs headers (case-sensitive) - reuses the real
# Chow et al. (2022) profile already extracted for NonLinLoc/GrowClust3D
# (Week 6), just reformatted with the headers QuakeMigrate's read_vmodel
# expects, rather than re-extracted from the NetCDF data again.
vel2grid_in = os.path.join(PROJECT_DIR, "nonlinloc", "vel2grid_tvz.in")
vmodel_rows = []
with open(vel2grid_in) as f:
    for line in f:
        if line.startswith("LAYER"):
            parts = line.split()
            depth, vp, vs = float(parts[1]), float(parts[2]), float(parts[4])
            vmodel_rows.append({"Depth": depth, "Vp": vp, "Vs": vs})

vmodel_df = pd.DataFrame(vmodel_rows)
vmodel_path = os.path.join(QM_LUT_DIR, "vmodel.csv")
vmodel_df.to_csv(vmodel_path, index=False)
print(f"Saved: {vmodel_path} ({len(vmodel_df)} depth levels)")
print(vmodel_df.head())

In [ ]:
# Grid specification: same region used consistently throughout the project
# (lat -40 to -37, lon 175-178, depth -3 to 40 km), same aeqd projection
# used for GrowClust3D.
#
# FIX: ll_corner/ur_corner must be given in GEOGRAPHIC coordinates
# (lon, lat, depth) - QuakeMigrate's LUT class does the projection into
# grid space internally itself, using the grid_proj/coord_proj pair.
# Confirmed directly from the documentation's own worked example, where
# ll_corner's longitude (116.075) matches the projection's lon_0 (116.75) -
# passing pre-projected km values (as done in the original attempt) fed
# invalid "latitude" values into the internal transform and produced NaNs.
cproj = Proj(proj="longlat", ellps="WGS84", datum="WGS84", no_defs=True)
gproj = Proj(proj="aeqd", lon_0=TARGET_LON, lat_0=TARGET_LAT, ellps="WGS84", units="km", no_defs=True)

LAT_MIN, LAT_MAX = -40.0, -37.0
LON_MIN, LON_MAX = 175.0, 178.0
DEPTH_MIN, DEPTH_MAX = -3.0, 40.0  # positive-down convention, matches NonLinLoc/GrowClust3D setup

NODE_SPACING_KM = 2.0

grid_spec = AttribDict()
grid_spec.ll_corner = [LON_MIN, LAT_MIN, DEPTH_MIN]
grid_spec.ur_corner = [LON_MAX, LAT_MAX, DEPTH_MAX]
grid_spec.node_spacing = [NODE_SPACING_KM, NODE_SPACING_KM, NODE_SPACING_KM]
grid_spec.grid_proj = gproj
grid_spec.coord_proj = cproj

# Rough expected grid size, for reference (exact count is computed
# internally by QuakeMigrate from the projected extent)
x_proj, y_proj = gproj([LON_MIN, LON_MAX], [LAT_MIN, LAT_MAX])
approx_n_x = int((max(x_proj) - min(x_proj)) / NODE_SPACING_KM) + 1
approx_n_y = int((max(y_proj) - min(y_proj)) / NODE_SPACING_KM) + 1
approx_n_z = int((DEPTH_MAX - DEPTH_MIN) / NODE_SPACING_KM) + 1
print(f"Approximate grid dimensions: {approx_n_x} x {approx_n_y} x {approx_n_z} "
      f"= {approx_n_x*approx_n_y*approx_n_z:,} nodes")

import pickle
grid_spec_path = os.path.join(QM_LUT_DIR, "grid_spec.pkl")
with open(grid_spec_path, "wb") as f:
    pickle.dump(grid_spec, f)
print(f"Saved: {grid_spec_path}")